# Modeling: ALS recommender + MLflow

Цель: построить рекомендательную систему для e-commerce с фокусом на событие `addtocart`.

В ноутбуке реализованы этапы:
1. Формализация задачи и метрик.
2. Подготовка событий и temporal split.
3. Baseline (popularity).
4. ALS (implicit feedback) и сравнение с baseline.
5. Интерпретация результатов и ограничения.

In [1]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import RANDOM_STATE, TOP_K, TRAIN_TIME_QUANTILE, ROLLING_TIME_QUANTILES
from src.data import load_events, temporal_split, select_active_users, build_test_ground_truth
from src.train import run_single_fold

print("Project root:", ROOT)
print("TOP_K:", TOP_K)
print("deployment quantile:", TRAIN_TIME_QUANTILE)

Project root: /home/mle-user/mle-ecommerce-recommendations
TOP_K: 10
deployment quantile: 0.85


In [2]:
# Шаг 1. Общая структура данных и временной split

events = load_events()
print("Rows:", len(events))
print("Users:", events["visitorid"].nunique())
print("Items:", events["itemid"].nunique())
print("\nEvent distribution:\n", events["event"].value_counts())

train, test = temporal_split(events, quantile=TRAIN_TIME_QUANTILE)
train = select_active_users(train, max_users=80_000)
ground_truth = build_test_ground_truth(test)

print("\nTrain rows:", len(train))
print("Test rows:", len(test))
print("Users with add-to-cart in test:", len(ground_truth))

Rows: 2756101
Users: 1407580
Items: 235061



Event distribution:
 event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64



Train rows: 853038
Test rows: 413416
Users with add-to-cart in test: 6039


### Подвывод 1

- Логи событий сильно разреженные: `view` доминирует, целевой `addtocart` редкий.
- Валидация должна быть временной (train в прошлом, test в будущем), что мы и делаем через temporal split.
- На следующем шаге сравним baseline popularity, ALS и двухэтапный ALS+rereanker на одном и том же окне.

In [3]:
# Шаг 2. Сравнение baseline / ALS / ALS+rereanker на deployment-window

baseline_metrics, als_metrics, rerank_metrics, _ = run_single_fold(
    events,
    quantile=TRAIN_TIME_QUANTILE,
    seed=RANDOM_STATE,
    fit_reranker=True,
)

comparison = pd.DataFrame(
    [baseline_metrics, als_metrics, rerank_metrics],
    index=["baseline_popularity", "als", "als_reranker"],
)
comparison

/home/mle-user/mle-ecommerce-recommendations/venv/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

,recall@10,precision@10,map@10,hit_rate@10,coverage@10,eval_users
baseline_popularity,0.038721,0.007407,0.040741,0.074074,0.000102,27
als,0.037037,0.003704,0.009259,0.037037,0.002265,27
als_reranker,0.000000,0.000000,0.000000,0.000000,0.005988,1


### Подвывод 2

- В этом запуске baseline лучше по recall/precision/map/hit_rate;
- ALS даёт более высокую coverage;
- Нужно увеличить eval_users и стабилизировать оценку (rolling folds), прежде чем делать вывод о победителе.

In [4]:
# Шаг 3. Rolling-time validation (устойчивость по времени)

rolling_rows = []
for i, q in enumerate(ROLLING_TIME_QUANTILES, start=1):
    b, a, r, _ = run_single_fold(events, quantile=q, seed=RANDOM_STATE + i, fit_reranker=True)
    rolling_rows.extend([
        {"fold": i, "quantile": q, "model": "baseline", **b},
        {"fold": i, "quantile": q, "model": "als", **a},
        {"fold": i, "quantile": q, "model": "als_reranker", **r},
    ])

rolling_df = pd.DataFrame(rolling_rows)
rolling_df[["fold", "quantile", "model", "recall@10", "map@10", "hit_rate@10", "coverage@10"]]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

,fold,quantile,model,recall@10,map@10,hit_rate@10,coverage@10
0,1,0.75,baseline,0.000000,0.000000,0.000000,0.000106
1,1,0.75,als,0.000000,0.000000,0.000000,0.002374
2,1,0.75,als_reranker,0.000000,0.000000,0.000000,0.005928
3,2,0.85,baseline,0.062735,0.066667,0.125000,0.000102
4,2,0.85,als,0.062500,0.015046,0.083333,0.001937
5,2,0.85,als_reranker,0.125000,0.125000,1.000000,0.006667


In [5]:
# Средние метрики по фолдам
rolling_df.groupby("model")[["recall@10", "map@10", "hit_rate@10", "coverage@10"]].mean().sort_values("recall@10", ascending=False)

,recall@10,map@10,hit_rate@10,coverage@10
model,,,,
als_reranker,0.062500,0.062500,0.500000,0.006297
baseline,0.031368,0.033333,0.062500,0.000104
als,0.031250,0.007523,0.041667,0.002156


### Подвывод 3

- Rolling validation показывает, стабильны ли улучшения модели на разных временных окнах.
- Если в одном фолде метрика лучше, а в другом хуже — важна не разовая победа, а средняя устойчивость.
- В проде решение о деплое принимает не ноутбук, а DAG champion/challenger gating (по `recall@10`).

## Итог по моделированию

1. Построен двухэтапный пайплайн: **ALS candidate generation + reranker**.
2. В reranker включены признаки из item metadata (категории и популярность).
3. Добавлена rolling-time validation для более честной оценки устойчивости.
4. Артефакты и метрики логируются в MLflow и используются в Airflow DAG для автоматического решения о промоуте модели.